# F3_01_Preprocesamiento

    ## Avance Fase 3 – Semana 2

    **Proyecto:** Análisis global de la industria fitness y gimnasios 2000-2026  
    **Propósito:** preparar, transformar, validar y exportar el dataset que será utilizado por los notebooks de algoritmos, mediciones y núcleo POO.

    Este notebook responde al requerimiento de preprocesamiento y transformación del dataset para la S3. 
    Toma como entrada `data/raw/clean_gym_data.csv` y genera `data/processed/gym_data_processed.csv`.

    ## Resultado esperado

    Al finalizar la ejecución debe existir un dataset procesado con columnas limpias, tipos corregidos, nulos tratados, variables derivadas, periodos temporales, variables normalizadas y validaciones finales aprobadas.

> Nota técnica: este notebook está diseñado para ejecutarse desde cualquier subcarpeta del repositorio. 
> Las rutas se resuelven buscando automáticamente la carpeta raíz `gym-fitness-analytics`.

## 1. Importación de librerías

Se importan `Path`, `pandas` y `numpy`, necesarias para rutas robustas, manipulación tabular y tratamiento de valores especiales.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

## 2. Configuración robusta de rutas

Esta celda busca la raíz del proyecto y define rutas de entrada y salida. Resultado esperado: impresión de rutas correctas.

In [ ]:
def encontrar_raiz_proyecto(nombre_repo="gym-fitness-analytics"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_repo:
            return ruta

    raise FileNotFoundError(
        f"No se pudo encontrar la raíz del proyecto '{nombre_repo}' desde {ruta_actual}"
    )


PROJECT_ROOT = encontrar_raiz_proyecto()

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "clean_gym_data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "gym_data_processed.csv"

print("Directorio actual:", Path.cwd().resolve())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta raw:", RAW_PATH)
print("Ruta processed:", PROCESSED_PATH)

## 3. Carga del dataset raw

Carga el dataset original. Resultado esperado: cantidad de filas, columnas y primeras filas.

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset raw en la ruta: {RAW_PATH}"
    )

df_raw = pd.read_csv(RAW_PATH)

print("Dataset raw cargado correctamente.")
print("Filas:", df_raw.shape[0])
print("Columnas:", df_raw.shape[1])

df_raw.head()

## 4. Exploración inicial

Revisa dimensiones, tipos, nulos, duplicados y estadísticos. Sirve como evidencia previa a las transformaciones.

In [ ]:
print("Dimensiones del dataset raw:")
print(df_raw.shape)

print("\nTipos de datos iniciales:")
display(df_raw.dtypes)

print("\nValores nulos por columna:")
display(df_raw.isna().sum())

print("\nDuplicados exactos:")
print(df_raw.duplicated().sum())

print("\nResumen estadístico general:")
display(df_raw.describe(include="all").T)

## 5. Normalización de nombres de columnas

Convierte columnas a minúsculas y reemplaza espacios/guiones por `_`. Resultado esperado: nombres homogéneos.

In [ ]:
def normalizar_nombres_columnas(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    return df


df = normalizar_nombres_columnas(df_raw)

print("Columnas normalizadas:")
display(df.columns.tolist())

## 6. Validación de columnas requeridas

Verifica que existan las columnas necesarias para la fase 3. Si falta alguna, se detiene la ejecución.

In [ ]:
columnas_requeridas = [
    "country",
    "region",
    "year",
    "gym_memberships",
    "fitness_participation_rate",
    "total_health_club_revenue_usd",
    "number_of_gyms",
    "gym_penetration_rate",
    "urban_population_percentage",
    "obesity_rate",
    "gdp_per_capita_usd",
    "population_total",
    "average_membership_cost_usd",
    "insufficient_physical_activity_pct"
]


def validar_columnas_requeridas(df, columnas):
    faltantes = [col for col in columnas if col not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

    return True


validar_columnas_requeridas(df, columnas_requeridas)

print("Todas las columnas requeridas están presentes.")

## 7. Conversión de tipos

Convierte texto y columnas numéricas. Resultado esperado: variables cuantitativas como numéricas y `year` como entero nullable.

In [ ]:
def convertir_tipos(df):
    df = df.copy()

    columnas_texto = ["country", "region"]

    columnas_numericas = [
        "year",
        "gym_memberships",
        "fitness_participation_rate",
        "total_health_club_revenue_usd",
        "number_of_gyms",
        "gym_penetration_rate",
        "urban_population_percentage",
        "obesity_rate",
        "gdp_per_capita_usd",
        "population_total",
        "average_membership_cost_usd",
        "insufficient_physical_activity_pct"
    ]

    for columna in columnas_texto:
        df[columna] = df[columna].astype("string").str.strip()

    for columna in columnas_numericas:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    df["year"] = df["year"].astype("Int64")

    return df


df = convertir_tipos(df)

print("Tipos de datos después de la conversión:")
display(df.dtypes)

## 8. Manejo de valores nulos

Texto se rellena con `Unknown`. Numéricos se imputan por mediana regional y, si aún quedan nulos, por mediana global.

In [ ]:
def manejar_valores_nulos(df):
    df = df.copy()

    columnas_texto = ["country", "region"]

    columnas_numericas = [
        "gym_memberships",
        "fitness_participation_rate",
        "total_health_club_revenue_usd",
        "number_of_gyms",
        "gym_penetration_rate",
        "urban_population_percentage",
        "obesity_rate",
        "gdp_per_capita_usd",
        "population_total",
        "average_membership_cost_usd",
        "insufficient_physical_activity_pct"
    ]

    for columna in columnas_texto:
        df[columna] = df[columna].fillna("Unknown")

    if df["year"].isna().any():
        mediana_year = int(df["year"].median())
        df["year"] = df["year"].fillna(mediana_year).astype("Int64")

    for columna in columnas_numericas:
        df[columna] = df.groupby("region")[columna].transform(
            lambda serie: serie.fillna(serie.median())
        )

        df[columna] = df[columna].fillna(df[columna].median())

    return df


nulos_antes = df.isna().sum()

df = manejar_valores_nulos(df)

nulos_despues = df.isna().sum()

comparacion_nulos = pd.DataFrame({
    "nulos_antes": nulos_antes,
    "nulos_despues": nulos_despues
})

comparacion_nulos

## 9. Duplicados

Elimina duplicados exactos. Resultado esperado: duplicados después igual a 0.

In [ ]:
duplicados_antes = df.duplicated().sum()

df = df.drop_duplicates().copy()

duplicados_despues = df.duplicated().sum()

print("Duplicados antes:", duplicados_antes)
print("Duplicados después:", duplicados_despues)

## 10. Corrección de valores negativos

Valores negativos no válidos se convierten en nulos y se imputan nuevamente.

In [ ]:
def corregir_valores_negativos(df):
    df = df.copy()

    columnas_no_negativas = [
        "gym_memberships",
        "fitness_participation_rate",
        "total_health_club_revenue_usd",
        "number_of_gyms",
        "gym_penetration_rate",
        "urban_population_percentage",
        "obesity_rate",
        "gdp_per_capita_usd",
        "population_total",
        "average_membership_cost_usd",
        "insufficient_physical_activity_pct"
    ]

    resumen_negativos = {}

    for columna in columnas_no_negativas:
        cantidad_negativos = int((df[columna] < 0).sum())
        resumen_negativos[columna] = cantidad_negativos

        if cantidad_negativos > 0:
            df.loc[df[columna] < 0, columna] = np.nan
            df[columna] = df.groupby("region")[columna].transform(
                lambda serie: serie.fillna(serie.median())
            )
            df[columna] = df[columna].fillna(df[columna].median())

    return df, resumen_negativos


df, resumen_negativos = corregir_valores_negativos(df)

df_negativos = pd.DataFrame(
    resumen_negativos.items(),
    columns=["columna", "negativos_detectados"]
)

df_negativos

## 11. Variables derivadas

Crea `memberships_per_100k`, `gyms_per_100k` y `revenue_per_membership_usd`.

In [ ]:
def crear_variables_derivadas(df):
    df = df.copy()

    df["memberships_per_100k"] = (
        df["gym_memberships"] / df["population_total"]
    ) * 100000

    df["gyms_per_100k"] = (
        df["number_of_gyms"] / df["population_total"]
    ) * 100000

    df["revenue_per_membership_usd"] = (
        df["total_health_club_revenue_usd"] / df["gym_memberships"]
    )

    columnas_derivadas = [
        "memberships_per_100k",
        "gyms_per_100k",
        "revenue_per_membership_usd"
    ]

    for columna in columnas_derivadas:
        df[columna] = df[columna].replace([np.inf, -np.inf], np.nan)
        df[columna] = df[columna].fillna(0)

    return df


df = crear_variables_derivadas(df)

df[
    [
        "country",
        "region",
        "year",
        "memberships_per_100k",
        "gyms_per_100k",
        "revenue_per_membership_usd"
    ]
].head()

## 12. Periodo temporal

Crea `periodo`: `pre_covid`, `covid`, `post_covid`.

In [ ]:
def asignar_periodo(df):
    df = df.copy()

    def clasificar_periodo(year):
        if year < 2020:
            return "pre_covid"
        elif year <= 2021:
            return "covid"
        else:
            return "post_covid"

    df["periodo"] = df["year"].apply(clasificar_periodo)

    return df


df = asignar_periodo(df)

df["periodo"].value_counts()

## 13. Normalización Min-Max

Crea columnas con sufijo `_norm` para futuras comparaciones y algoritmos de similitud.

In [ ]:
def normalizar_minmax(df, columnas):
    df = df.copy()

    for columna in columnas:
        minimo = df[columna].min()
        maximo = df[columna].max()

        if pd.isna(minimo) or pd.isna(maximo):
            df[f"{columna}_norm"] = 0
        elif maximo == minimo:
            df[f"{columna}_norm"] = 0
        else:
            df[f"{columna}_norm"] = (
                (df[columna] - minimo) / (maximo - minimo)
            )

    return df


columnas_a_normalizar = [
    "gym_memberships",
    "fitness_participation_rate",
    "total_health_club_revenue_usd",
    "number_of_gyms",
    "gym_penetration_rate",
    "urban_population_percentage",
    "obesity_rate",
    "gdp_per_capita_usd",
    "population_total",
    "average_membership_cost_usd",
    "insufficient_physical_activity_pct",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd"
]

df = normalizar_minmax(df, columnas_a_normalizar)

columnas_norm = [col for col in df.columns if col.endswith("_norm")]

print("Cantidad de columnas normalizadas:", len(columnas_norm))

df[columnas_norm].head()

## 14. Validación final

Evalúa filas, duplicados, nulos, variables derivadas, no negatividad y rangos normalizados.

In [ ]:
def validar_dataset_final(df):
    validaciones = {}

    validaciones["filas_mayor_que_cero"] = df.shape[0] > 0
    validaciones["sin_duplicados"] = df.duplicated().sum() == 0
    validaciones["sin_nulos_totales"] = df.isna().sum().sum() == 0

    columnas_requeridas_finales = columnas_requeridas + [
        "memberships_per_100k",
        "gyms_per_100k",
        "revenue_per_membership_usd",
        "periodo"
    ]

    for columna in columnas_requeridas_finales:
        validaciones[f"existe_{columna}"] = columna in df.columns

    columnas_no_negativas = [
        "gym_memberships",
        "total_health_club_revenue_usd",
        "number_of_gyms",
        "population_total",
        "memberships_per_100k",
        "gyms_per_100k",
        "revenue_per_membership_usd"
    ]

    for columna in columnas_no_negativas:
        validaciones[f"{columna}_no_negativa"] = (df[columna] >= 0).all()

    columnas_norm = [col for col in df.columns if col.endswith("_norm")]

    for columna in columnas_norm:
        validaciones[f"{columna}_entre_0_y_1"] = (
            (df[columna] >= 0).all() and (df[columna] <= 1).all()
        )

    periodos_esperados = {"pre_covid", "covid", "post_covid"}
    periodos_obtenidos = set(df["periodo"].dropna().unique())

    validaciones["periodos_validos"] = periodos_obtenidos.issubset(periodos_esperados)

    return validaciones


validaciones_finales = validar_dataset_final(df)

df_validaciones = pd.DataFrame(
    validaciones_finales.items(),
    columns=["validacion", "resultado"]
)

df_validaciones

In [ ]:
validaciones_fallidas = df_validaciones[df_validaciones["resultado"] == False]

if not validaciones_fallidas.empty:
    display(validaciones_fallidas)
    raise ValueError("Existen validaciones finales fallidas.")

print("Todas las validaciones finales fueron aprobadas.")

## 15. Exportación

Exporta el dataset procesado a `data/processed/gym_data_processed.csv`.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_PATH, index=False)

print("Dataset procesado exportado correctamente.")
print("Ruta de salida:", PROCESSED_PATH)
print("Filas exportadas:", df.shape[0])
print("Columnas exportadas:", df.shape[1])

## 16. Resumen técnico

Genera un resumen usable en el informe y README.

In [ ]:
resumen_pipeline = {
    "archivo_entrada": str(RAW_PATH),
    "archivo_salida": str(PROCESSED_PATH),
    "filas_finales": df.shape[0],
    "columnas_finales": df.shape[1],
    "duplicados_finales": int(df.duplicated().sum()),
    "nulos_finales": int(df.isna().sum().sum()),
    "variables_derivadas": [
        "memberships_per_100k",
        "gyms_per_100k",
        "revenue_per_membership_usd",
        "periodo"
    ],
    "columnas_normalizadas": len(columnas_norm)
}

resumen_pipeline

## Conclusión

Este notebook deja preparado el dataset base para los notebooks algorítmicos, de mediciones y POO de Fase 3. Con esto se cubre el apartado de preprocesamiento y transformación del dataset solicitado en S3.